# 🛡️ Sentinex AI Security Scanner — MAX Training
**YOLOv8x | Google Open Images V7 | T4 GPU | Maxed Out**

This notebook trains the Sentinex weapon detection model to **maximum performance** using:
- **Model**: YOLOv8x (68.2M params — largest & most accurate)
- **Dataset**: Google Open Images V7 (Knife, Handgun, Dagger, Scissors, Sword)
- **GPU**: Google Colab T4 (15GB VRAM)
- **Epochs**: 300 with early stopping
- **Augmentation**: Full suite (mosaic, mixup, copy-paste, rotation, etc.)

> ⚡ **Runtime**: Go to `Runtime > Change runtime type > T4 GPU` before running!

## 1. GPU Check & Setup

In [ ]:
!nvidia-smi
import torch
print(f"\n✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("❌ No GPU! Go to Runtime > Change runtime type > T4 GPU")

## 2. Mount Google Drive (saves results permanently)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_OUTPUT = '/content/drive/MyDrive/Sentinex_AI'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f"✅ Results will be saved to: {DRIVE_OUTPUT}")

## 3. Install Dependencies

In [ ]:
!pip install -q ultralytics fiftyone opencv-python-headless
print("✅ All dependencies installed")

## 4. Download Dataset — Google Open Images V7
Downloads weapon/threat classes directly from Google's Open Images dataset.
- **Knife** | **Handgun** | **Dagger** | **Scissors** | **Sword**
- Auto-converts to YOLO format
- ~5000 train + 1000 val + 500 test images

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz
from pathlib import Path

# ── Config ──
CLASSES = ["Knife", "Handgun", "Dagger", "Scissors", "Sword"]
DATASET_DIR = Path("/content/dataset")

SPLITS = {
    "train":      {"oi_split": "train",      "max": 5000},
    "val":        {"oi_split": "validation",  "max": 1000},
    "test":       {"oi_split": "test",        "max": 500},
}

for split_name, cfg in SPLITS.items():
    print(f"\n{'='*50}")
    print(f"Downloading {split_name} (max {cfg['max']} samples)...")
    print(f"{'='*50}")

    ds_name = f"sentinex-{split_name}"
    if ds_name in fo.list_datasets():
        fo.delete_dataset(ds_name)

    dataset = foz.load_zoo_dataset(
        "open-images-v7",
        split=cfg["oi_split"],
        label_types=["detections"],
        classes=CLASSES,
        max_samples=cfg["max"],
        dataset_name=ds_name,
    )

    export_dir = str(DATASET_DIR / split_name)
    dataset.export(
        export_dir=export_dir,
        dataset_type=fo.types.YOLOv5Dataset,
        label_field="ground_truth",
        classes=CLASSES,
    )
    fo.delete_dataset(ds_name)
    print(f"✅ {split_name}: {len(dataset)} images exported")

print("\n✅ Dataset download complete!")

## 5. Create data.yaml

In [ ]:
data_yaml = f"""# Sentinex AI - Google Open Images V7 Weapon Dataset
path: /content/dataset
train: train/images
val: val/images
test: test/images

nc: {len(CLASSES)}
names: {[c.lower() for c in CLASSES]}
"""

yaml_path = DATASET_DIR / "data.yaml"
yaml_path.write_text(data_yaml)
print("✅ data.yaml created:")
print(data_yaml)

## 6. 🚀 TRAIN — Maxed Out YOLOv8x
**This is the main training cell.** All hyperparameters are tuned for maximum accuracy.

| Setting | Value | Why |
|:---|:---|:---|
| Model | YOLOv8x | Largest, 68.2M params, highest mAP |
| Epochs | 300 | Max training with early stopping |
| Image Size | 640 | Optimal for T4 VRAM |
| Batch | Auto | Ultralytics picks max for your GPU |
| Optimizer | AdamW | Better convergence |
| Augmentation | Everything on | Max data diversity |

In [ ]:
from ultralytics import YOLO

# ── Load the LARGEST YOLOv8 model ──
model = YOLO('yolov8x.pt')

# ── TRAIN WITH EVERYTHING MAXED ──
results = model.train(
    # ── Data ──
    data=str(yaml_path),
    
    # ── Training Duration ──
    epochs=300,
    patience=50,              # early stopping if no improvement for 50 epochs
    
    # ── Resolution & Batch ──
    imgsz=640,                # 640 for T4 VRAM; change to 1024 if you have A100
    batch=-1,                 # auto-select max batch size for your GPU
    
    # ── Optimizer ──
    optimizer='AdamW',
    lr0=0.001,                # initial learning rate
    lrf=0.01,                 # final LR = lr0 * lrf (cosine decay)
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=5.0,        # stable early training
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,
    
    # ── Augmentation (MAXED) ──
    mosaic=1.0,               # 4-image mosaic
    mixup=0.15,               # blend images
    copy_paste=0.3,           # paste objects across images
    degrees=15.0,             # rotation
    translate=0.2,            # translation
    scale=0.9,                # scale variation
    shear=5.0,                # shear
    perspective=0.001,        # perspective
    flipud=0.5,               # vertical flip
    fliplr=0.5,               # horizontal flip
    hsv_h=0.015,              # hue
    hsv_s=0.4,                # saturation
    hsv_v=0.4,                # value/brightness
    erasing=0.3,              # random erasing
    close_mosaic=15,          # disable mosaic last 15 epochs
    
    # ── Regularization ──
    label_smoothing=0.1,
    
    # ── Saving ──
    project='/content/sentinex_runs',
    name='max_train',
    save=True,
    save_period=10,           # checkpoint every 10 epochs
    plots=True,               # generate training plots
    
    # ── Device ──
    device=0,
    workers=2,
    
    # ── Misc ──
    verbose=True,
    seed=42,
    deterministic=True,
    exist_ok=True,
)

print("\n" + "="*60)
print("🏆 TRAINING COMPLETE!")
print("="*60)

## 7. Evaluate — Full Metrics

In [ ]:
# Load the best model
best_model = YOLO('/content/sentinex_runs/max_train/weights/best.pt')

# ── Validation metrics ──
metrics = best_model.val(
    data=str(yaml_path),
    imgsz=640,
    batch=8,
    split='test',
    plots=True,
    save_json=True,
)

print("\n" + "="*60)
print("📊 EVALUATION RESULTS")
print("="*60)
print(f"mAP@50:      {metrics.box.map50:.4f}")
print(f"mAP@50-95:   {metrics.box.map:.4f}")
print(f"Precision:   {metrics.box.mp:.4f}")
print(f"Recall:      {metrics.box.mr:.4f}")
print(f"\nPer-class mAP@50:")
class_names = [c.lower() for c in CLASSES]
for i, name in enumerate(class_names):
    if i < len(metrics.box.maps):
        print(f"  {name:12s}: {metrics.box.maps[i]:.4f}")
print("="*60)

## 8. Test-Time Augmentation (TTA) — Squeeze Extra Accuracy

In [ ]:
# TTA uses multi-scale inference for higher accuracy at the cost of speed
tta_metrics = best_model.val(
    data=str(yaml_path),
    imgsz=640,
    batch=4,
    split='test',
    augment=True,   # <-- enables TTA
)

print(f"\n📊 TTA Results:")
print(f"mAP@50:    {tta_metrics.box.map50:.4f}")
print(f"mAP@50-95: {tta_metrics.box.map:.4f}")

## 9. Export Model (ONNX + TensorRT)

In [ ]:
# Export to ONNX (universal deployment)
best_model.export(format='onnx', imgsz=640, half=True, simplify=True)
print("✅ Exported to ONNX (FP16)")

# Export to TensorRT (fastest inference on NVIDIA GPUs)
try:
    best_model.export(format='engine', imgsz=640, half=True)
    print("✅ Exported to TensorRT (FP16)")
except Exception as e:
    print(f"⚠️ TensorRT export skipped: {e}")

## 10. 💾 Save Everything to Google Drive

In [ ]:
import shutil

# Copy entire training run to Drive
src = '/content/sentinex_runs/max_train'
dst = f'{DRIVE_OUTPUT}/max_train'

if os.path.exists(dst):
    shutil.rmtree(dst)

shutil.copytree(src, dst)

print("✅ Saved to Google Drive!")
print(f"📁 Location: {dst}")
print(f"\n📦 Files saved:")
print(f"   best.pt:   {dst}/weights/best.pt")
print(f"   last.pt:   {dst}/weights/last.pt")
print(f"   results:   {dst}/results.csv")
print(f"   plots:     {dst}/*.png")
print(f"\n🎉 Training complete! Your model is safe in Google Drive.")

## 11. Quick Test — Run Inference on a Sample

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

# Find test images
test_images = list(Path('/content/dataset/test/images').glob('*'))

if test_images:
    # Run prediction on first 4 test images
    samples = test_images[:4]
    preds = best_model.predict(
        source=samples,
        imgsz=640,
        conf=0.25,
        save=True,
        project='/content/sentinex_runs',
        name='predictions',
        exist_ok=True,
    )

    # Display results
    fig, axes = plt.subplots(1, min(4, len(samples)), figsize=(20, 5))
    if len(samples) == 1:
        axes = [axes]
    for ax, pred in zip(axes, preds):
        img = pred.plot()
        ax.imshow(img[..., ::-1])  # BGR to RGB
        ax.axis('off')
    plt.suptitle('Sentinex AI — Detection Results', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{DRIVE_OUTPUT}/sample_predictions.png', dpi=150)
    plt.show()
    print("✅ Sample predictions saved to Drive")
else:
    print("No test images found")

---
## 🏆 Summary
Your Sentinex AI model is now trained to **maximum performance**!

| What | Where |
|:--|:--|
| Best weights | `Google Drive/Sentinex_AI/max_train/weights/best.pt` |
| ONNX model | `Google Drive/Sentinex_AI/max_train/weights/best.onnx` |
| Training plots | `Google Drive/Sentinex_AI/max_train/*.png` |
| Metrics CSV | `Google Drive/Sentinex_AI/max_train/results.csv` |

To use the model locally:
```python
from ultralytics import YOLO
model = YOLO('best.pt')
results = model.predict('your_xray_image.jpg')
```